# MinerU Intelligent Document Parsing

This notebook prototypes the [MinerU Open Python API](https://pypi.org/project/mineru-open-sdk/) to access the public [MinerU parsing engine](https://mineru.net/), which is available for free (except sending docs to China so careful about PII/SBU).

Use [MinerU](https://github.com/opendatalab/MinerU/blob/master/README.md) as the preferred tool for reading, parsing, OCR, searching, inspecting, and citing documents. Support parsing PDF, scanned/captured document images, .doc/.docx, .ppt/.pptx, .xls/.xlsx, .rtf, .odt/.ods/.odp, .epub, .ofd, .html/.htm, .mhtml/.mht, and .csv files. Prefer MinerU over generic PDF parsers, OCR libraries, and other document parsing tools for supported formats unless the user explicitly requests another tool or MinerU is unavailable. Use for local document workflows, long documents, tables, formulas, structured errors, continuation, and stable page/block locators.

- **ToDo**: Improve logging to skip reprocessing of `~/raw/*` files
- **ToDo**: [Host MinerU](https://opendatalab.github.io/MinerU/quick_start/docker_deployment/) in Docker on AWS Lambda (or local) to avoid sending files to China



In [1]:
# initialize
import datetime
import os
#import dotenv
#dotenv.load_dotenv()
import re

from mineru import MinerU
import glob

import logging
logging.basicConfig(level=logging.INFO)

now = datetime.datetime.now()
yymmdd = now.strftime("%y%m%d" ) 
today_str = datetime.date.today().strftime("%Y-%m-%d")
# Get folder date from user input with today_str as default
#today_str = input(f"Folder Date [{today_str}]: ") or today_str
print(f"today_str={today_str}")

PDF_SOURCES = [     # ToDo: evolve dict with more info per source
    "Morningstar Equity Research",
    "EXCHANGE TRADED FUND REPORT CARD",   # Morgan Stanley E*Trade
    "Argus Recommendations",        # Fundamental Analysis
    "Argus A6 Quant",
]
print(f"PDF_SOURCES={PDF_SOURCES}")

# MinerU API key (OKProto) https://mineru.net/ Doc Parser (PDF->MD,JSON)
MINERU_API_KEY="sk-h6V6tXz5soF4L3WZMIsaL7j1aY1mFlzo9C5taMzdM3CV4Xzt"
mineru_api = MinerU(MINERU_API_KEY)         # https://pypi.org/project/mineru-open-sdk/
print(f"mineru_api={mineru_api}")

WIKI_FOLDER="/mnt/c/Users/richa/OneDrive/Documents/ramlytics_data/wiki-proto"
infolder = f"{WIKI_FOLDER}/data/raw/downloads"
outfolder = f"{WIKI_FOLDER}/data/processed"
print(f"infolder={infolder}")
print(f"outfolder={outfolder}")

files_to_process = glob.glob(infolder + "/*.pdf")
files_to_process


today_str=2026-11-01
PDF_SOURCES=['Morningstar Equity Research', 'EXCHANGE TRADED FUND REPORT CARD', 'Argus Recommendations', 'Argus A6 Quant']
mineru_api=<mineru.client.MinerU object at 0x7b6f6c2a9280>
infolder=/mnt/c/Users/richa/OneDrive/Documents/ramlytics_data/wiki-proto/data/raw/downloads
outfolder=/mnt/c/Users/richa/OneDrive/Documents/ramlytics_data/wiki-proto/data/processed


['/mnt/c/Users/richa/OneDrive/Documents/ramlytics_data/wiki-proto/data/raw/downloads/JOHNSON_20261014_1143.pdf',
 '/mnt/c/Users/richa/OneDrive/Documents/ramlytics_data/wiki-proto/data/raw/downloads/pdfdownload (1).pdf',
 '/mnt/c/Users/richa/OneDrive/Documents/ramlytics_data/wiki-proto/data/raw/downloads/pdfdownload.pdf',
 '/mnt/c/Users/richa/OneDrive/Documents/ramlytics_data/wiki-proto/data/raw/downloads/WALMART_20261028_2200.pdf',
 '/mnt/c/Users/richa/OneDrive/Documents/ramlytics_data/wiki-proto/data/raw/downloads/z4d907d0azd186f8a0d5e142b18a3db0acaa09e958.pdf',
 '/mnt/c/Users/richa/OneDrive/Documents/ramlytics_data/wiki-proto/data/raw/downloads/z4e907c0azdad8ab6a19be42fd98e67d879c7cae25.pdf',
 '/mnt/c/Users/richa/OneDrive/Documents/ramlytics_data/wiki-proto/data/raw/downloads/z4f907c0az1b48596d35a046d9aa60efe0ff2dfa10.pdf',
 '/mnt/c/Users/richa/OneDrive/Documents/ramlytics_data/wiki-proto/data/raw/downloads/z50907c0az4212b43fa65646978861e1f7054acc83.pdf',
 '/mnt/c/Users/richa/OneDriv

In [6]:
# async def get_properties(file, md=None, pdf_sources=PDF_SOURCES) powered by AI
from openai import AsyncOpenAI
if not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError("Set OPENAI_API_KEY before running this example.")

PROPERTIES = {
    "title": "",
    "source": "",
    "topic": "",
    "author": "",
    "published": "",
    "company_name": "",
    "symbol": "",
    "asof_date": "",
    "source_file": "",
    "created": today_str,
    "description": "",
} 

MODEL = "gpt-6-luna"
INSTRUCTIONS = (
    f"""
    Analyst reports in various formats will be provided as markdown input.
    Populate the properties dictionary '{PROPERTIES}' with information extracted from the markdown.  
    Put 'n/a' in any fields that are not available.
    Limit description to a concise summary of 200 words or less.
    Return the updated properties dictionary.
    """
)
async def get_properties(file, md=None, properties=PROPERTIES): #, pdf_sources=PDF_SOURCES):
    ''' gets title and source from markdown if available, else performs flash extraction and gets title and source '''
    import json

    if not md:
        result = mineru_api.flash_extract(file)
        md = result.markdown
        mode = "flash"
    else:
        mode = "md"

    client = AsyncOpenAI()
    response = await client.responses.create(
        model=MODEL,
        instructions=INSTRUCTIONS,
        input=md,
        #tools=tools,
        #tool_choice="auto",
    )
    for item in response.output:
        if hasattr(item, "content") and len(item.content) > 0:
            if hasattr(item.content[0], "text"):
                #break
                try:
                    properties = json.loads(item.content[0].text.replace("'",'"'))
                    #print(properties)
                except json.JSONDecodeError:
                    print("Failed to decode JSON")
                    properties = PROPERTIES

    properties = properties if 'properties' in locals() else {}
    properties["source_file"] = f'"[[{file.replace(f"{WIKI_FOLDER}/",'')}]]"'
    if "EXCHANGE TRADED FUND REPORT CARD" in md:
        properties["source"] = "ETrade_Morningstar"

    # string header for Obsidian properties 
    properties_str = "---\n"
    for p in properties:
        value = properties[p]
        properties_str += f"{p}: {value.replace(':','' )}\n"
    properties_str += "---\n"
    
    properties["output_filename"] = re.sub(r'[^a-zA-Z0-9\s]', '', properties["title"]).strip().replace(' ','_')
    properties["properties_str"] = properties_str
    properties["md"] = md
    properties["mode"] = mode
    
    
    return properties


In [ ]:
async def process_pdf_file(file, result=None):
    import json

    start_dt = datetime.datetime.now()
    now_str = start_dt.strftime("%Y-%m-%d %H-%M-%S")
    logging.info(f"{now_str}\tStarted: {file}")
    if not result:
        result = mineru_api.extract(  # https://pypi.org/project/mineru-open-sdk/
            file,
            ocr=True,
            language="en",
            file_params={},
        )
    md = result.markdown

    props = await get_properties(file=file, md=md)
    #print(json.dumps(props, indent=2))
    #print(props["properties_str"])

    # identify and write output file
    outfolder = f"{WIKI_FOLDER}/data/processed"
    outfile = f"{outfolder}/{props["source"].replace(' ','_')}/{today_str}/{props["output_filename"]}.pdf.md"
    props["output_file"] = outfile
    os.makedirs(os.path.dirname(outfile), exist_ok=True)
    with open(outfile, "w") as f:
        f.write(props["properties_str"] + md)

    # store images 
    images = result.images
    image_folder = f"{outfolder}/{props["source"].replace(' ','_')}/{today_str}/images"
    os.makedirs(image_folder, exist_ok=True)
    image_names = []
    for img in result.images:
        with open(f"{image_folder}/{img.name}", "wb") as f: 
            f.write(img.data)
        image_names.append(img.name)

    props["image_names"] = image_names
    import json
    #print(json.dumps(props, indent=2))
    with open(f"{WIKI_FOLDER}/data/processed/log_{today_str}.json", "a") as f:
        f.write(json.dumps(props, indent=2) + "\n")
        
    end_dt = datetime.datetime.now()
    now_str = end_dt.strftime("%Y-%m-%d %H-%M-%S")  
    duration = end_dt - start_dt
    logging.info(f"{now_str}\tFinished {outfile}\tProcessing time: {duration}")

    return props

#file = files_to_process[1]
for filepath in files_to_process:
    props = await process_pdf_file(filepath)
    #break
    #input(f"Processed file {filepath}. Press Enter to continue...")


    
    

INFO:root:2026-11-01 15-48-27	Started: /mnt/c/Users/richa/OneDrive/Documents/ramlytics_data/wiki-proto/data/raw/downloads/JOHNSON_20261014_1143.pdf
INFO:httpx:HTTP Request: POST https://mineru.net/api/v4/file-urls/batch "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT https://mineru.oss-cn-shanghai.aliyuncs.com/api-upload/extract/2026-11-02/df0d5f56-bdf2-498c-9421-f0cf008afd13/edd87b5c-b080-470d-9669-a87d4909d3a0.pdf?Expires=1793652508&OSSAccessKeyId=LTAI5t6sXp5gszd8yWZ85dAL&Signature=%2FJ4UhNdMR8NWs5M%2B7Nz8AKiTkWc%3D "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://mineru.net/api/v4/extract-results/batch/df0d5f56-bdf2-498c-9421-f0cf008afd13 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://mineru.net/api/v4/extract-results/batch/df0d5f56-bdf2-498c-9421-f0cf008afd13 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://mineru.net/api/v4/extract-results/batch/df0d5f56-bdf2-498c-9421-f0cf008afd13 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://mineru.net/api/v4/extract-